# 🧭 Séance 2 — Mini‑projet Titanic : de NumPy à un premier fichier CSV
_Master IA‑GI — Python pour les sciences de données_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelallioui/python-for-ds-s02/blob/main/mini-projet-02/mini-projet-02.ipynb)

**Comment ouvrir ce notebook :** sur votre ordinateur, ouvrez-le **depuis son dossier** `mini-projet-02/`, sans le déplacer. Le fichier de départ est dans `data/`, à côté ; **vos résultats** sont écrits dans `result/`, à côté aussi. NumPy doit être installé (`pip install numpy` ; déjà présent avec Anaconda).

**Dans Colab** (le bouton ci-dessus) : Colab n'ouvre que ce notebook, sans son dossier. Exécutez d'abord, dans une cellule, ces deux lignes :
```
!git clone https://github.com/yelallioui/python-for-ds-s02
%cd python-for-ds-s02/mini-projet-02
```
Pour **garder votre travail**, faites **Fichier ▸ Enregistrer une copie dans Drive** : Colab ne sauvegarde pas un notebook ouvert depuis GitHub.

**Objectifs** — à la fin de ce notebook, vous saurez :
- **charger** les 7 colonnes numériques de `titanic.csv` dans un tableau NumPy `data` (décimaux, `nan` pour les trous), puis en **extraire** des colonnes typées
- **interroger** les tableaux sans boucle : masques combinés, tableau voyage seul × classe, agrégats par axe, fonctions qui ignorent les `nan`
- **calculer** de nouvelles colonnes avec les opérations vectorisées, les fonctions universelles et le broadcasting ; **trier**, **repérer** et **tirer au hasard** de façon reproductible
- **écrire** un nouveau fichier CSV, `s2_titanic.csv`, avec `np.savetxt`, puis **constater les limites de NumPy** : le point de départ de la séance 3 (Pandas)

**Durée estimée** : 45 minutes environ · **Parties** :
- **M** · Mini-projet : le Titanic en tableaux

**Comment travailler seul**
1. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
2. Lisez le *cours en bref*, puis **exécutez le code complet** : c'est lui qui contient les détails des diapos.
3. Faites chaque exercice en remplaçant les `...` par votre code, puis exécutez la cellule : un ✅ ou un ❌ s'affiche.
4. Un ❌ ? Ouvrez l'**indice 1**, puis 2, puis 3.

### 🧰 Pour commencer : importer, puis se contrôler
Exécutez la cellule ci-dessous. Elle importe ce qu'il faut et définit **`verifier`**, la fonction qui affiche ✅ ou ❌ sous chaque exercice. Rien de nouveau dans son écriture : une fonction (`def`), un `try` / `except` et un `lambda`, vus en séance 1.

In [ ]:
import csv
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)

def verifier(ex, description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (même si le test provoque une erreur)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + ex + " · " + description)

### 🪪 Qui êtes-vous ?
Écrivez votre **nom**, votre **prénom** et votre **CNE** **entre guillemets**, en remplaçant les `...` (par exemple `nom = "Benali"`). Ces trois lignes permettent à l'enseignant de reconnaître votre travail : le contrôle reste ❌ tant qu'elles ne sont pas remplies. **Ne partagez votre dépôt qu'en privé.**

In [ ]:
nom = ...
prenom = ...
cne = ...

print(nom, prenom, cne)
verifier("identité", "nom, prénom et CNE renseignés", lambda: type(nom) is str and type(prenom) is str and type(cne) is str and len(nom.strip()) >= 2 and len(prenom.strip()) >= 2 and len(cne.strip()) >= 8)

**Le travail de data science commence ici.** La séance 1 était une séance de révision : ses résultats ne sont pas repris. Le mini‑projet part du fichier brut `titanic.csv`, et **chaque mini‑projet, à partir d'aujourd'hui, produit un nouveau fichier CSV** que le suivant reprend.

**La question qui traverse les six séances : qui a survécu au naufrage du Titanic, et pourquoi ?** À la dernière séance, vous y répondrez avec des chiffres, des tests et une conclusion honnête. Aujourd'hui, **vous poussez NumPy à fond** sur les **7 colonnes numériques** du fichier, puis vous **constatez où il s'arrête** : un tableau NumPy n'a qu'un seul type, donc le sexe, le nom, le port ou la cabine n'y entrent pas. C'est ce qui amène Pandas, à la séance 3.

**Le fil du mini‑projet**
- **Charger et typer** : **M1** charger les 7 colonnes numériques dans `data` · **M2** en extraire des tableaux typés
- **Interroger** : **M3** masques et tableau voyage seul × classe · **M4** les âges manquants · **M5** la boucle contre le tableau
- **Fabriquer** : **M6** quatre colonnes calculées · **M7** standardiser par broadcasting
- **Explorer** : **M8** trier et repérer · **M9** tirer au hasard, de façon reproductible
- **Résultat et limites** : **M10** écrire `result/s2_titanic.csv`, puis **cinq questions que NumPy ne sait pas traiter**
- **Fichier de départ** : `data/titanic.csv`, à côté de ce notebook : **c'est à vous de le charger** (M1). _Aucun geste nouveau : tout vient de la partie 1 et du cours (la transposée `.T`, étape M7, est dans le support de cours)._

**🧭 Étape M1 — Charger les colonnes numériques de titanic.csv**

**Comment charger les colonnes numériques de `data/titanic.csv`**, le fichier de départ, à côté de ce notebook, dans un tableau NumPy ? Mettez dans `data` un **tableau NumPy** de **891 lignes** (un passager par ligne) et **7 colonnes**, dans l'ordre de la liste `colonnes` (fournie), tout en **décimaux** (`float`) : une case vide devient un **`nan`**. Les 5 colonnes de texte (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`) ne peuvent pas entrer dans le même tableau : un `ndarray` n'a qu'un seul type. _(`np.genfromtxt` lirait ce fichier d'un coup, mais la virgule des noms, `Braund, Mr. Owen Harris`, décale les colonnes : on passe par `csv`.)_

<details>
<summary>💡 Indice 1 · la piste</summary>

Vous savez lire un CSV (séance 1) : `csv.DictReader` rend une liste de dictionnaires, un par passager. Pour un passager `l` et une colonne `c`, `l[c]` est du texte.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Une case vide ne se convertit pas en nombre : `float(l[c]) if l[c] != "" else np.nan`. Une compréhension sur les colonnes, dans une compréhension sur les passagers, donne une liste de listes, que `np.array(...)` transforme en tableau 2-D.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`with open("data/titanic.csv", encoding="utf-8", newline="") as f:` puis, dans le bloc, `lignes = list(csv.DictReader(f))` ; ensuite `data = np.array([[float(l[c]) if l[c] != "" else np.nan for c in colonnes] for l in lignes])`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
colonnes = ["PassengerId", "Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]      # les 7 colonnes numériques du fichier

# votre code : ouvrir le fichier, puis remplir data
data = ...

verifier("M1", "data : un tableau NumPy de 891 lignes et 7 colonnes de décimaux", lambda: type(data) is np.ndarray and data.shape == (891, 7) and data.dtype == np.float64)
verifier("M1", "le premier passager : numéro 1, mort, 3e classe, 22 ans, 1 frère ou conjoint, 0 parent, tarif 7,25", lambda: list(data[0]) == [1, 0, 3, 22, 1, 0, 7.25])
verifier("M1", "les 177 âges manquants sont des nan, et ce sont les seuls", lambda: np.isnan(data[:, 3]).sum() == 177 and np.isnan(data).sum() == 177)

_À partir d'ici, la cellule ci‑dessous refait ce que vous venez d'écrire (`colonnes` et `data`) : les étapes suivantes repartent d'une base correcte, même si l'étape M1 vous a résisté._

In [ ]:
colonnes = ["PassengerId", "Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
with open("data/titanic.csv", encoding="utf-8", newline="") as f:
    lignes = list(csv.DictReader(f))
data = np.array([[float(l[c]) if l[c] != "" else np.nan for c in colonnes] for l in lignes])
print(data.shape)

**🧭 Étape M2 — Redonner leur type aux colonnes**

Extrayez sept tableaux, une colonne de `data` chacun : `ids`, `survived`, `pclass`, `sibsp` et `parch` en **entiers** ; `age` et `fare` restent des **décimaux** (les `nan` de l'âge restent des `nan`).

<details>
<summary>💡 Indice 1 · la piste</summary>

Une colonne se lit avec une tranche : `data[:, 0]` est la colonne `PassengerId` (`print(list(enumerate(colonnes)))` donne les numéros). Une tranche est une **vue** : elle partage les données de `data`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`astype(int)` convertit une colonne en entiers (et crée une copie). L'âge et le tarif sont déjà des décimaux : une simple tranche suffit.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`ids = data[:, 0].astype(int)` ; `age = data[:, 3]` ; `fare = data[:, 6]` ; et de même pour `survived` (colonne 1), `pclass` (2), `sibsp` (4) et `parch` (5).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ids = ...
survived = ...
pclass = ...
age = ...
sibsp = ...
parch = ...
fare = ...

verifier("M2", "ids : entiers de 1 à 891", lambda: np.issubdtype(ids.dtype, np.integer) and ids.shape == (891,) and ids.min() == 1 and ids.max() == 891)
verifier("M2", "pclass et survived : entiers, de sommes 2057 et 342", lambda: np.issubdtype(pclass.dtype, np.integer) and pclass.sum() == 2057 and np.issubdtype(survived.dtype, np.integer) and survived.sum() == 342)
verifier("M2", "sibsp et parch : entiers, de sommes 466 et 340", lambda: np.issubdtype(sibsp.dtype, np.integer) and sibsp.sum() == 466 and np.issubdtype(parch.dtype, np.integer) and parch.sum() == 340)
verifier("M2", "age : 891 valeurs float, dont 177 nan", lambda: age.shape == (891,) and age.dtype == np.float64 and np.isnan(age).sum() == 177)
verifier("M2", "fare : float, maximum 512,33", lambda: fare.dtype == np.float64 and abs(fare.max() - 512.3292) < 1e-6)

_Les étapes suivantes repartent de tableaux corrects : la cellule ci‑dessous reconstruit ceux de l'étape M2, que vous ayez réussi ou non._

In [ ]:
ids = data[:, 0].astype(int)
survived = data[:, 1].astype(int)
pclass = data[:, 2].astype(int)
age = data[:, 3]
sibsp = data[:, 4].astype(int)
parch = data[:, 5].astype(int)
fare = data[:, 6]
print("tableaux prêts :", age.shape, "avec", int(np.isnan(age).sum()), "âges manquants")

**🧭 Étape M3 — Masques combinés : un tableau voyage seul × classe**

Un passager **voyage seul** s'il n'a ni frère, sœur, conjoint, ni parent, ni enfant à bord : `seul_m` est le masque de ces passagers (fourni). Six groupes : les passagers seuls et accompagnés de chaque classe. Avec **deux masques combinés** par groupe :
1. `effectifs_sc` : un tableau **2 × 3** des effectifs (lignes : seuls, accompagnés ; colonnes : 1re, 2e, 3e classe)
2. `survie_sc` : un tableau **2 × 3** des **taux de survie**
3. `par_classe` et `par_groupe` : les effectifs totaux par classe et par groupe (seuls, accompagnés), avec **`sum(axis=...)`**

_C'est un travail de six filtres écrits à la main : Pandas le fera en une ligne, à la séance 4._

<details>
<summary>💡 Indice 1 · la piste</summary>

Un groupe est le **et** de deux masques : `ms & (pclass == c)` (les parenthèses sont obligatoires). Un masque se compte avec `sum()` ; une proportion de survivants est la moyenne de `survived` sur le groupe.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

La compréhension parcourt les six groupes ; `.reshape(2, 3)` les range en deux lignes de trois. `axis=0` additionne les lignes (une valeur par classe), `axis=1` additionne les colonnes (une valeur par groupe).

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`effectifs_sc = np.array([(ms & (pclass == c)).sum() for ms in (seul_m, ~seul_m) for c in (1, 2, 3)]).reshape(2, 3)` ; `survie_sc = np.array([survived[ms & (pclass == c)].mean() for ...]).reshape(2, 3)` ; `par_classe = effectifs_sc.sum(axis=0)` ; `par_groupe = effectifs_sc.sum(axis=1)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
seul_m = (sibsp + parch) == 0                                  # fourni : True pour un passager seul

effectifs_sc = np.array([... for ms in (seul_m, ~seul_m) for c in (1, 2, 3)]).reshape(2, 3)
survie_sc = np.array([... for ms in (seul_m, ~seul_m) for c in (1, 2, 3)]).reshape(2, 3)
par_classe = ...
par_groupe = ...

verifier("M3", "effectifs : 109, 104, 324 passagers seuls et 107, 80, 167 accompagnés", lambda: np.array_equal(effectifs_sc, [[109, 104, 324], [107, 80, 167]]))
verifier("M3", "survie : 53 %, 35 %, 21 % des seuls ; 73 %, 64 %, 30 % des accompagnés", lambda: np.allclose(survie_sc, [[0.5321, 0.3462, 0.2130], [0.7290, 0.6375, 0.2994]], atol=0.001))
verifier("M3", "totaux par classe : 216, 184, 491", lambda: np.array_equal(par_classe, [216, 184, 491]))
verifier("M3", "totaux par groupe : 537 passagers seuls, 354 accompagnés", lambda: np.array_equal(par_groupe, [537, 354]))

**🧭 Étape M4 — Les âges manquants : ce qu'ils font aux calculs**

1. `age_moyen_naif` : la moyenne de `age` **sans précaution** (que vaut-elle ?)
2. `n_trous` : le nombre de `nan` dans `age`
3. `age_moyen` et `age_median` : la moyenne et la médiane des âges **connus**
4. `age_median_classe` : l'âge **médian de chaque classe** `[1re, 2e, 3e]`, en ignorant les `nan`

_Aucun âge n'est rempli ici : décider quoi faire des trous est le travail de la séance 3, avec Pandas._

<details>
<summary>💡 Indice 1 · la piste</summary>

Un seul `nan` contamine un calcul : `age.mean()` rend `nan`. Un `nan` se repère avec `np.isnan`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`np.nanmean` et `np.nanmedian` ignorent les `nan`. Pour une classe, on applique la fonction au masque : `np.nanmedian(age[pclass == c])`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`age_moyen_naif = age.mean()` ; `n_trous = np.isnan(age).sum()` ; `age_moyen = np.nanmean(age)` ; `age_median = np.nanmedian(age)` ; `age_median_classe = [np.nanmedian(age[pclass == c]) for c in (1, 2, 3)]`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
age_moyen_naif = ...
n_trous = ...
age_moyen = ...
age_median = ...
age_median_classe = ...

verifier("M4", "la moyenne naïve est contaminée : nan", lambda: np.isnan(age_moyen_naif))
verifier("M4", "177 âges manquants", lambda: n_trous == 177)
verifier("M4", "âge moyen : 29,70 ans ; âge médian : 28 ans", lambda: abs(age_moyen - 29.6991) < 0.001 and age_median == 28.0)
verifier("M4", "âge médian par classe : 37, 29, 24 ans", lambda: np.allclose(age_median_classe, [37, 29, 24]))

**🧭 Étape M5 — La boucle de la séance 1 contre le tableau**

`survie_boucle` est le calcul de la séance 1 : **une boucle** sur les 891 lignes de `data` pour chaque classe. Dans `survie_np` : le **même résultat** avec un masque par classe. Le chronomètre compare ensuite les deux écritures.

<details>
<summary>💡 Indice 1 · la piste</summary>

Un masque par classe, puis `mean()` sur `survived`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`survived[pclass == c].mean()` pour la classe `c` ; une compréhension sur `(1, 2, 3)` évite de répéter.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`survie_np = [survived[pclass == c].mean() for c in (1, 2, 3)]`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
import time


def chrono(f):
    """Temps d'exécution de f(), en secondes."""
    debut = time.perf_counter()
    f()
    return time.perf_counter() - debut


survie_boucle = []
for c in (1, 2, 3):
    n = s = 0
    for ligne in data:
        if ligne[2] == c:                      # colonne 2 : la classe
            n += 1
            s += ligne[1]                      # colonne 1 : survécu (0 ou 1)
    survie_boucle.append(s / n)

survie_np = ...

verifier("M5", "NumPy donne les mêmes taux que la boucle : 63 %, 47 %, 24 %", lambda: np.allclose(survie_np, survie_boucle) and np.allclose(survie_np, [0.630, 0.473, 0.242], atol=0.001))

def boucle():
    for c in (1, 2, 3):
        sum(ligne[1] for ligne in data if ligne[2] == c) / sum(1 for ligne in data if ligne[2] == c)

t_boucle = chrono(boucle)
t_np = chrono(lambda: [survived[pclass == c].mean() for c in (1, 2, 3)])
print(f"boucle : {t_boucle * 1000:.2f} ms · tableau : {t_np * 1000:.3f} ms")

**🧭 Étape M6 — Quatre colonnes calculées, sans boucle**

Pour **chaque passager**, en opérations vectorisées :
1. `taille_famille` : lui‑même + ses frères, sœurs et conjoint + ses parents et enfants, un tableau d'entiers
2. `seul` : **1** s'il voyage seul (`taille_famille` vaut 1), **0** sinon, un tableau d'entiers
3. `fare_par_personne` : le tarif divisé par la taille de la famille
4. `log_fare` : le **logarithme naturel** du tarif **plus un** (`np.log`, série 4) : il comprime les très gros tarifs

<details>
<summary>💡 Indice 1 · la piste</summary>

Les opérations arithmétiques entre tableaux de même forme se font case par case. Une comparaison rend des booléens, que `astype(int)` transforme en 0 et 1.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`np.log(x)` est le logarithme naturel ; comme toute fonction universelle, elle s'applique à tout le tableau d'un coup. Le tarif peut valoir 0 : on prend le log de `fare + 1`, comme à l'exercice 4.3.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`taille_famille = sibsp + parch + 1` ; `seul = (taille_famille == 1).astype(int)` ; `fare_par_personne = fare / taille_famille` ; `log_fare = np.log(fare + 1)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
taille_famille = ...
seul = ...
fare_par_personne = ...
log_fare = ...

verifier("M6", "taille_famille : des entiers, 2 pour le passager 1, 11 au maximum", lambda: np.issubdtype(taille_famille.dtype, np.integer) and taille_famille[0] == 2 and taille_famille.max() == 11)
verifier("M6", "seul : des 0 et des 1, et 537 passagers voyagent seuls", lambda: np.issubdtype(seul.dtype, np.integer) and set(np.unique(seul)) == {0, 1} and seul.sum() == 537)
verifier("M6", "fare_par_personne : 3,625 pour le passager 1, 512,33 au maximum", lambda: abs(fare_par_personne[0] - 3.625) < 1e-9 and abs(fare_par_personne.max() - 512.3292) < 1e-6)
verifier("M6", "log_fare : 2,1102 pour le passager 1", lambda: abs(log_fare[0] - 2.1102) < 1e-3)

**🧭 Étape M7 — Standardiser par broadcasting**

1. `X` : un tableau de forme **`(891, 5)`** : les colonnes `fare`, `pclass`, `sibsp`, `parch`, `taille_famille` côte à côte
2. `X_z` : chaque colonne **centrée** (moyenne 0) et **réduite** (écart‑type 1), en **une seule ligne**, par broadcasting

_Ce tableau n'est pas enregistré : en apprentissage automatique, les moyennes et les écarts‑types se calculent sur les données d'entraînement seules (séance 6)._

<details>
<summary>💡 Indice 1 · la piste</summary>

Un tableau construit avec les cinq colonnes a la forme `(5, 891)` : il faut échanger lignes et colonnes. L'attribut `.T` (la transposée) le fait : **geste nouveau**.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`X.mean(axis=0)` donne une valeur par colonne, de forme `(5,)` ; soustraire ce `(5,)` à `X` de forme `(891, 5)` se fait par broadcasting.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`X = np.array([fare, pclass, sibsp, parch, taille_famille]).T` ; `X_z = (X - X.mean(axis=0)) / X.std(axis=0)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
taille_famille = sibsp + parch + 1                                      # fourni (étape M6)
X = ...
X_z = ...

verifier("M7", "X a la forme (891, 5)", lambda: X.shape == (891, 5))
verifier("M7", "X_z : chaque colonne a une moyenne nulle et un écart-type de 1", lambda: X_z.shape == (891, 5) and np.allclose(X_z.mean(axis=0), 0, atol=1e-9) and np.allclose(X_z.std(axis=0), 1))
verifier("M7", "le tarif du passager 1, standardisé : -0,5024", lambda: abs(X_z[0, 0] + 0.5024) < 1e-3)

**🧭 Étape M8 — Trier et repérer**

1. `ordre` : les **positions** qui rangent les passagers du tarif le **plus cher au moins cher** (à tarifs égaux, on garde l'ordre du fichier)
2. `top10_ids` : les **identifiants** des dix passagers qui ont payé le plus cher
3. `positions_nulles` : les positions des passagers dont le **tarif est nul** ; `n_tarifs_nuls` : leur nombre

_Les billets à zéro livre ne sont pas des erreurs de saisie évidentes : à examiner en séance 3._

<details>
<summary>💡 Indice 1 · la piste</summary>

`np.argsort(x)` donne les positions qui trient `x` du plus petit au plus grand. Pour l'ordre inverse, on trie l'opposé : `np.argsort(-fare)`. L'option `kind="stable"` garde l'ordre d'origine entre égaux.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Une liste de positions choisit des valeurs : `ids[ordre[:10]]`. `np.where(condition)[0]` donne les positions où la condition est vraie.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`ordre = np.argsort(-fare, kind="stable")` ; `top10_ids = ids[ordre[:10]]` ; `positions_nulles = np.where(fare == 0)[0]` ; `n_tarifs_nuls = len(positions_nulles)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ordre = ...
top10_ids = ...
positions_nulles = ...
n_tarifs_nuls = ...

verifier("M8", "ordre : 891 positions, le billet le plus cher est celui du passager 259", lambda: ordre.shape == (891,) and ids[ordre[0]] == 259)
verifier("M8", "les dix plus chers : 259, 680, 738, 28, 89, 342, 439, 312, 743, 119", lambda: list(top10_ids) == [259, 680, 738, 28, 89, 342, 439, 312, 743, 119])
verifier("M8", "les dix plus chers voyageaient tous en première classe", lambda: (pclass[ordre[:10]] == 1).all())
verifier("M8", "15 passagers ont un tarif nul", lambda: n_tarifs_nuls == 15 and len(positions_nulles) == 15 and (fare[positions_nulles] == 0).all())

**🧭 Étape M9 — Tirer au hasard, de façon reproductible**

Un taux calculé sur un **échantillon** varie d'un tirage à l'autre. Avec un générateur de **graine 42** :
1. `rng` : le générateur ; `echantillon` : **100 positions** tirées **sans remise** parmi les 891 passagers
2. `taux_echantillon` : le taux de survie de ces 100 passagers ; `taux_global` : le taux de survie des 891
3. `taux_1000` : un tableau des taux de survie de **1000 échantillons** de 100 passagers

_L'écart entre les taux d'échantillon est l'incertitude d'un chiffre calculé sur peu de données : la séance 6 apprendra à la mesurer._

<details>
<summary>💡 Indice 1 · la piste</summary>

`np.random.default_rng(graine)` crée un générateur : la même graine redonne les mêmes tirages. `rng.choice(n, size=k, replace=False)` tire `k` positions parmi `n`, sans doublon.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Un taux de survie, c'est la moyenne de `survived` sur les positions tirées : `survived[positions].mean()`. Pour 1000 échantillons, une compréhension qui tire à chaque tour avec le même `rng`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`rng = np.random.default_rng(42)` ; `echantillon = rng.choice(891, size=100, replace=False)` ; `taux_echantillon = survived[echantillon].mean()` ; `taux_global = survived.mean()` ; `taux_1000 = np.array([survived[rng.choice(891, size=100, replace=False)].mean() for _ in range(1000)])`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
rng = ...
echantillon = ...
taux_echantillon = ...
taux_global = ...
taux_1000 = ...

verifier("M9", "100 positions distinctes parmi 891", lambda: echantillon.shape == (100,) and len(set(echantillon)) == 100 and echantillon.max() < 891)
verifier("M9", "reproductible : la même graine redonne le même tirage", lambda: np.array_equal(echantillon, np.random.default_rng(42).choice(891, size=100, replace=False)))
verifier("M9", "taux global : 38,4 % ; taux de l'échantillon : proche mais pas égal", lambda: abs(taux_global - 0.3838) < 0.001 and 0.25 < taux_echantillon < 0.5)
verifier("M9", "1000 taux : centrés sur le taux global, dispersés d'environ 4,5 points", lambda: taux_1000.shape == (1000,) and abs(taux_1000.mean() - taux_global) < 0.01 and 0.03 < taux_1000.std() < 0.06)

**🧭 Étape M10 — Le résultat de la séance : un nouveau fichier CSV**

Écrivez **`s2_titanic.csv`** : les **sept colonnes numériques plus les quatre colonnes calculées** (`taille_famille`, `seul`, `fare_par_personne`, `log_fare`), soit 11 colonnes. Les noms (`en_tete`), les formats de chaque colonne (`formats`) et le `chemin` (qui désigne le dossier `result`) sont fournis. Rassemblez les 11 colonnes dans **un tableau `sortie`** (891 lignes, 11 colonnes), puis écrivez-le avec **`np.savetxt`**, comme en série 8. Les âges manquants s'écrivent `nan`. La séance 3 (Pandas) reprendra ce fichier.

<details>
<summary>💡 Indice 1 · la piste</summary>

Un tableau construit avec les onze colonnes a la forme `(11, 891)` : `.T` l'échange en `(891, 11)`, comme à l'étape M7.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`np.savetxt(chemin, sortie, delimiter=",", fmt=formats, header=en_tete, comments="")` : `fmt` accepte une liste, un format par colonne.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`sortie = np.array([ids, survived, pclass, age, sibsp, parch, fare, taille_famille, seul, fare_par_personne, log_fare]).T` puis `np.savetxt(chemin, sortie, delimiter=",", fmt=formats, header=en_tete, comments="")`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
taille_famille = sibsp + parch + 1                                           # fourni (étape M6)
seul = (taille_famille == 1).astype(int)
fare_par_personne = fare / taille_famille
log_fare = np.log(fare + 1)
colonnes_sortie = colonnes + ["taille_famille", "seul", "fare_par_personne", "log_fare"]
en_tete = "PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,taille_famille,seul,fare_par_personne,log_fare"      # les 11 noms, séparés par des virgules
formats = ["%d", "%d", "%d", "%.2f", "%d", "%d", "%.4f", "%d", "%d", "%.4f", "%.4f"]      # un format par colonne
Path("result").mkdir(exist_ok=True)                                          # le dossier result, créé s'il manque
chemin = Path("result") / "s2_titanic.csv"

sortie = ...                              # votre code : les 11 colonnes, en un tableau de 891 lignes
# votre code : écrivez sortie dans chemin avec np.savetxt


def relire():
    with open(chemin, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


verifier("M10", "891 lignes et 11 colonnes : les 7 numériques, puis les 4 calculées", lambda: len(relire()) == 891 and list(relire()[0]) == colonnes_sortie)
verifier("M10", "les valeurs sont conservées : passager 1, 3e classe, 22 ans, tarif 7.25 ; les 177 âges manquants s'écrivent nan", lambda: relire()[0]["PassengerId"] == "1" and relire()[0]["Pclass"] == "3" and relire()[0]["Age"] == "22.00" and relire()[0]["Fare"] == "7.2500" and sum(1 for r in relire() if r["Age"] == "nan") == 177)
verifier("M10", "passager 1 : taille_famille 2, seul 0, fare_par_personne 3.625", lambda: relire()[0]["taille_famille"] == "2" and relire()[0]["seul"] == "0" and relire()[0]["fare_par_personne"] == "3.6250")
verifier("M10", "log_fare du passager 1 : 2,1102 ; 537 passagers seuls", lambda: abs(float(relire()[0]["log_fare"]) - 2.1102) < 1e-3 and sum(int(r["seul"]) for r in relire()) == 537)

**🎉 Votre premier fichier CSV est prêt.** Vous avez `s2_titanic.csv` : les **sept colonnes numériques**, avec **quatre colonnes calculées avec NumPy**, les âges manquants écrits `nan`. Il est dans le dossier `result/` de votre mini‑projet.

---
## 🚧 Ce que NumPy ne sait pas faire
Vous avez poussé NumPy à fond sur **7 colonnes numériques**. Mais le fichier en a **12** : le sexe, le nom, le ticket, la cabine et le port n'ont pas pu entrer dans `data`, car **un tableau NumPy n'a qu'un seul type**. Voici cinq questions sur le Titanic qui restent **sans réponse avec NumPy seul**. Réfléchissez à ce qui bloque, sans écrire de code : ce sont celles que Pandas traitera à la séance 3.

1. **Les femmes ont-elles mieux survécu que les hommes ?** Le sexe est du texte : il n'est pas dans `data`.
2. **Combien de passagers ont embarqué dans chaque port ?** `Embarked` est du texte, avec deux cases vides.
3. **Quel est le titre le plus fréquent (Mr, Mrs, Miss…) ?** Il est caché dans le texte de la colonne `Name`.
4. **Quelle colonne de `data` contient l'âge ?** Un tableau n'a pas de noms de colonnes : il faut compter les indices, ou garder la liste `colonnes` à côté.
5. **Remplacer chaque âge manquant par la médiane de sa classe et de son sexe ?** Il faudrait regrouper selon deux critères, dont un est du texte, puis réécrire les valeurs dans le bon groupe.

In [ ]:
print("les noms des colonnes sont à part :", colonnes)
print("un seul type pour tout le tableau :", data.dtype)
print("data[:3, 4] :", data[:3, 4], "→ c'est SibSp : le sexe n'a pas de colonne")

**À la séance 3**, votre enseignant vous remettra le dossier `seance-03`, dont le mini‑projet contient ce fichier **et `titanic.csv` brut** : Pandas les chargera, les réunira sur `PassengerId`, retrouvera les cinq colonnes de texte, **auditera** les trous (177 âges, 687 cabines, 2 ports, 15 tarifs nuls) et **décidera** quoi en faire.